In [ ]:
# Example 1: Public API surface — Keccak256 class + auto-loaded backend.
from eth_hash.auto import keccak
from eth_hash.main import Keccak256
from eth_hash.abc import BackendAPI, PreImageAPI

print(f"keccak repr: {keccak!r}")
print(f"Keccak256 class: {Keccak256}")
print(f"BackendAPI: {BackendAPI}")
print(f"PreImageAPI: {PreImageAPI}")

assert isinstance(keccak, Keccak256)
assert hasattr(keccak, 'new')
assert callable(keccak)


In [ ]:
# Example 2: Known-answer test — empty input keccak-256.
# Ethereum-flavored keccak (NOT NIST SHA3) of b'' is the well-known constant
# 0xc5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470.
from eth_hash.auto import keccak

digest = keccak(b'')
expected_hex = 'c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470'

print(f"keccak(b'') = {digest.hex()}")
print(f"expected    = {expected_hex}")
print(f"len(digest) = {len(digest)} bytes")

assert isinstance(digest, bytes)
assert len(digest) == 32
assert digest.hex() == expected_hex


In [ ]:
# Example 3: Hash a real Solidity function selector.
# transfer(address,uint256) selector is the first 4 bytes of
# keccak256('transfer(address,uint256)') = 0xa9059cbb...
from eth_hash.auto import keccak

signature = b'transfer(address,uint256)'
digest = keccak(signature)
selector = digest[:4]

print(f"signature: {signature!r}")
print(f"full digest: {digest.hex()}")
print(f"4-byte selector: 0x{selector.hex()}")

assert selector.hex() == 'a9059cbb'


In [ ]:
# Example 4: Incremental hashing via the preimage / .new() API.
# .new() returns a PreImage object you can .update() then .digest().
from eth_hash.auto import keccak

one_shot = keccak(b'hello world')

hasher = keccak.new(b'hello ')
hasher.update(b'world')
incremental = hasher.digest()

print(f"one-shot:    {one_shot.hex()}")
print(f"incremental: {incremental.hex()}")

assert one_shot == incremental
assert len(incremental) == 32


In [ ]:
# Example 5: iterate every importable backend in SUPPORTED_BACKENDS and check they agree on keccak(b'').
# Install one with pip install "eth-hash[pycryptodome]".
from eth_hash.main import Keccak256
from eth_hash.backends import SUPPORTED_BACKENDS

print(f"SUPPORTED_BACKENDS = {SUPPORTED_BACKENDS}")

# Try to construct a Keccak256 with each importable backend and verify
# they all agree on the empty-string digest.
expected = bytes.fromhex('c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470')
results = {}
for name in SUPPORTED_BACKENDS:
    try:
        mod = __import__(f'eth_hash.backends.{name}', fromlist=['backend'])
        k = Keccak256(mod.backend)
        results[name] = k(b'')
    except ImportError as e:
        results[name] = f'unavailable: {e}'

for name, val in results.items():
    print(f"  {name}: {val.hex() if isinstance(val, bytes) else val}")

good = [v for v in results.values() if isinstance(v, bytes)]
assert all(v == expected for v in good), 'backend disagreement on keccak(b"")'
assert good, 'no backend produced a digest'
